# 🚀 YouTube Advanced Downloader

Welcome to the multi-quality, multi-format YouTube downloader! 

## 📌 Features:
- **Video + Audio**: Download standard videos with sound.
- **Audio Only**: Extract just the sound (MP3).
- **Video Only**: Download just the video track, no audio.
- **Separate Audio & Video**: Download both tracks as separate files.
- **Selectable Resolutions**: Choose exact qualities like 4K, 2K, 1080p, 720p, 480p, etc.

### 🛠️ Step 1: Install Dependencies
Run the cell below to install `yt-dlp` which powers this notebook.

In [ ]:
!pip -q install -U yt-dlp
print('✅ yt-dlp installed successfully!')

### 📥 Step 2: Configure & Download
Fill out the form below. Paste your YouTube URL, select your preferred format and resolution, and run the cell!

In [ ]:
import yt_dlp
import os
from google.colab import files

# @title Download Settings
url = "" # @param {type:"string"}
download_type = "Video + Audio" # @param ["Video + Audio", "Audio Only", "Video Only (No Audio)", "Separate Video and Audio"]
video_quality = "1080p" # @param ["Best Available (4K/8K)", "1440p (2K)", "1080p", "720p", "480p", "360p", "Lowest Quality"]

if not url:
    print("❌ Please provide a YouTube URL.")
else:
    print(f"⏳ Processing {url}...")
    ydl_opts = {
        "outtmpl": "%(title)s.%(ext)s",
        "restrictfilenames": True,
    }

    # Determine max height for video
    height_map = {
        "Best Available (4K/8K)": None,
        "1440p (2K)": 1440,
        "1080p": 1080,
        "720p": 720,
        "480p": 480,
        "360p": 360,
        "Lowest Quality": "worst"
    }
    
    max_height = height_map.get(video_quality)
    
    # Construct format string
    if max_height == "worst":
        vid_format = "worstvideo"
        aud_format = "worstaudio"
        best_combo = "worst"
    elif max_height is None:
        vid_format = "bestvideo"
        aud_format = "bestaudio"
        best_combo = "best"
    else:
        vid_format = f"bestvideo[height<={max_height}]"
        aud_format = "bestaudio"
        best_combo = f"best[height<={max_height}]"

    if download_type == "Video + Audio":
        ydl_opts["format"] = f"{vid_format}[ext=mp4]+{aud_format}[ext=m4a]/{best_combo}[ext=mp4]/{best_combo}"
        ydl_opts["merge_output_format"] = "mp4"
            
    elif download_type == "Audio Only":
        # Audio only ignores video_quality, but we can use 'worst' if Lowest Quality is selected
        ydl_opts["format"] = aud_format
        ydl_opts["postprocessors"] = [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '64' if max_height == 'worst' else '192',
        }]
        
    elif download_type == "Video Only (No Audio)":
        ydl_opts["format"] = f"{vid_format}[ext=mp4]/{vid_format}"
            
    elif download_type == "Separate Video and Audio":
        ydl_opts["format"] = f"{vid_format},{aud_format}"
            
    print(f"\n🚀 Starting download with yt-dlp (Target Quality: {video_quality})...\n")
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
            print("\n✅ Download finished! The files will be saved in your Colab environment.")
    except Exception as e:
        print(f"\n❌ An error occurred: {e}")


### 🔍 Step 3: Advanced Format Selection (Optional)
If the above options don't cover your needs, you can list formats and download a specific one here. It will automatically use the URL you pasted in Step 2.

In [ ]:
# This will use the 'url' variable from Step 2
if 'url' in locals() and url:
    with yt_dlp.YoutubeDL() as ydl:
        info = ydl.extract_info(url, download=False)
        formats = info.get("formats", [])
        
        print("\n=== Available Formats ===\n")
        for f in formats:
            if f.get("ext") in ["mp4", "webm", "m4a"]:
                format_id = f.get("format_id")
                ext = f.get("ext")
                res = f.get("resolution", "audio only")
                note = f.get("format_note", "")
                vcodec = f.get("vcodec", "")
                acodec = f.get("acodec", "")
                fps = f.get("fps", "")
                filesize = f.get("filesize") or f.get("filesize_approx")
                
                size_str = f"{filesize/1024/1024:.2f} MB" if filesize else "Unknown size"
                fps_str = f"{fps}fps" if fps else ""
                
                type_str = ""
                if vcodec != "none" and acodec != "none":
                    type_str = "Video+Audio"
                elif vcodec != "none":
                    type_str = "Video Only"
                else:
                    type_str = "Audio Only"
                
                print(f"ID: {format_id: <4} | {ext: <4} | {res: <12} | {type_str: <12} | {size_str}")
else:
    print("❌ Please provide a YouTube URL in Step 2 first.")

In [ ]:
# @title Download Custom Format ID
format_id = "" # @param {type:"string"}

if 'url' in locals() and url and format_id:
    ydl_opts = {
        "format": format_id,
        "outtmpl": "%(title)s.%(ext)s",
        "restrictfilenames": True,
    }
    print(f"\n🚀 Downloading format {format_id}...\n")
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])
    print("\n✅ Custom download finished!")
else:
    print("❌ Please ensure URL is provided in Step 2 and format_id is entered here.")

### 💾 Step 4: Download Files to Local Device
Run this cell to zip and download everything to your computer.

In [ ]:
import glob
from google.colab import files

print("Gathering files...")
media_files = []
for ext in ["*.mp4", "*.mp3", "*.webm", "*.m4a", "*.mkv"]:
    media_files.extend(glob.glob(ext))
    
if media_files:
    for file in media_files:
        print(f"Downloading {file}...")
        files.download(file)
else:
    print("No downloaded media files found in the environment.")